# Drug Target Analysis: GPCRs

This notebook demonstrates a workflow from the README, focusing on G Protein-Coupled Receptors (GPCRs).

**Workflow:**
1. **UniProt API** - Retrieve human GPCR sequences
2. **Sequence Motif Search** - Find conserved GPCR motifs (DRY, NPxxY, CWxP, etc.)
3. **Structure Motif Search** - Analyze GPCR binding pockets in 3D structures
4. **py3Dmol Visualization** - Visualize identified binding pockets

---

## Setup

In [ ]:
import sys
import os
import pandas as pd
import json
import glob
import py3Dmol

# Add module paths from the scripts
sys.path.insert(0, os.path.abspath('../sequence_motif'))
sys.path.insert(0, os.path.abspath('../structure_motif'))

from file_converter import process_protein_files
from motif_searcher import run_motif_search
from uniprot_api import search_uniprot, to_csv
from search_3d_motif import search_single_file, parse_motif_file

# Directory setup
MOTIF_LIBRARIES_DIR = '../sequence_motif/motif_libraries'
STRUCTURE_MOTIFS_DIR = '../structure_motif/motifs'
OUTPUT_DIR = '../outputs/gpcr_analysis'
PROTEIN_FILES_DIR = '../protein_files'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup complete!')

---
## Step 1: Retrieve GPCR Sequences from UniProt

In [ ]:
# Sample GPCR sequences for demo (in case UniProt query fails)
SAMPLE_GPCRS = [
    {'Entry': 'P07550', 'protein_name': 'Beta-2 adrenergic receptor', 'sequence': 'MGQPGNGSAFLLAPNGSHAPDHDVTQQRDEVWVVGMGIVMSLIVLAIVFGNVLVITAIAKFERLQTVTNYFITSLACADLVMGLAVVPFGAAHILMKMWTFGNFWCEFWTSIDVLCVTASIETLCVIAVDRYFAITSPFKYQSLLTKNKARVIILMVWIVSGLTSFLPIQMHWYRATHQEAINCYANETCCDFFTNQAYAIASSIVSFYVPLVIMVFVYSRVFQEAKRQLQKIDKSEGRFHVQNLSQVEQDGRTGHGLRRSSKFCLKEHKALKTLGIIMGTFTLCWLPFFIVNIVHVIQDNLIRKEVYILLNWIGYVNSGFNPLIYCRSPDFRIAFQELLCLRRSSLKAYGNGYSSNSNGKTDYMGEASGCQLGQEKESERLCEDPPGTESFVNCQGTVPSLSLDSQGRNCSTNDSLL', 'organism_name': 'Homo sapiens'},
    {'Entry': 'P35348', 'protein_name': 'Alpha-1A adrenergic receptor', 'sequence': 'MVFLSGNASDSSNCTQPPAPVNISKAILLGVILGGLILFGVLGNILVILSVACHRHLHSVTHYYIVNLAVADLLLTSTVLPFSAIFEVLGYWAFGRVFCNIWAAVDVLCCTASIMGLCIISIDRYIGVSYPLRYPTIVTQRRGVRALLCVWVLSLVISIGPLFGWRQPAPEDETICQINEEPGYVLFSALGSFYLPLAIILVMYCRVYVVAKRESRGLKSGLKTDKSDSEQVTLRIHRKNAPAGGSGMASAKTKTHFSVRLLKFSREKKAAKTLGIVVGCFVLCWLPFFLVMPIGSFFPDFKPSETVFKIVFWLGYLNSCINPIIYPCSSQEFKKAFQNVLRIQCLCRKQSSKHALGYTLHPPSQAVEGQHKDMVRIPVGSRETFYRISKTDGVCEWKFFSSMPRGSARITVSKDQSSCTTARVRSKSFLQVCCCVGPSTPSLDKNHQVPTIKVHTISLSENGEEV', 'organism_name': 'Homo sapiens'},
    {'Entry': 'P08172', 'protein_name': 'Muscarinic acetylcholine receptor M2', 'sequence': 'MNNSTNSSNNSLALTSPYKTFEVVFIVLVAGSLSLVTIIGNILVMVSIKVNRHLQTVNNYFLFSLACADLIIGVFSMNLYTLYTVIGYWPLGPVVCDLWLALDYVVSNASVMNLLIISFDRYFCVTKPLTYPVKRTTKMAGMMIAAAWVLSFILWAPAILFWQFIVGVRTVEDGECYIQFFSNAAVTFGTAIAAFYLPVIIMTVLYWHISRASKSRIKKDKKEPVANQDPVSPSLVQGRIVKPNNNNMPSSDDGLEHNKIQNGKAPRDPVTENCVQGEEKESSNDSTSVSAKTKTVSGSSPAETENSPSSNSTVNPNETPRPVLVQQHTQMQEAAAPTAPRPANSVSKPSLQPSMEMSKSGTTLNNSCSQ', 'organism_name': 'Homo sapiens'},
    {'Entry': 'P21554', 'protein_name': 'Cannabinoid receptor 1', 'sequence': 'MKSILDGLADTTFRTITTDLLYVGSNDIQYEDIKGDMASKLGYFPQKFPLTSFRGSPFQEKMTAGDNPQLVPADQVNITEFYNKSLSSFKENEENIQCGENFMDIECFMVLNPSQQLAIAVLSLTLGTFTVLENLLVLCVILHSRSLRCRPSYHFIGSLAVADLLGSVIFVYSFIDFHVFHRKDSRNVFLFKLGGVTASFTASVGSLFLTAIDRYISIHRPLAYKRIVTRPKAVVAFCLMWTIAIVIAVLPLLGWNCEKLQSVCSDIFPHIDETYLMFWIGVTSVLLLFIVYAYMYILWKAHSHAVRMIQRGTQKSIIIHTSEDGKVQVTRPDQARMDIRLAKTLVLILVVLIICWGPLLAIMVYDVFGKMNKLIKTVFAFCSMLCLLNSTVNPIIYALRSKDLRHAFRSMFPSCEGTAQPLDNSMGDSDCLHKHANNAASVHRAAESCIKSTVKIAKVTMSVSTDTSAEAL', 'organism_name': 'Homo sapiens'}
]
print(f'Sample GPCR database: {len(SAMPLE_GPCRS)} sequences')

In [ ]:
# Try UniProt API first, fall back to sample data if that doesnt work for retrieval
gpcr_csv = os.path.join(OUTPUT_DIR, 'gpcrs.csv')

queries = [
    '(cc_function:"G-protein coupled receptor") AND (organism_id:9606)',
    '(keyword:"G-protein coupled receptor") AND (organism_id:9606)'
]

gpcr_df = None
for query in queries:
    print(f'Trying: {query}')
    try:
        data = search_uniprot(query, limit=10) ##limiting to 10 sequences
        lines = data.strip().split('\n')
        if len(lines) > 1:
            to_csv(data, gpcr_csv)
            if os.path.exists(gpcr_csv):
                gpcr_df = pd.read_csv(gpcr_csv)
                print(f'Success! Retrieved {len(gpcr_df)} sequences')
                break
    except Exception as e:
        print(f'  Query failed: {e}')
        continue

if gpcr_df is None or len(gpcr_df) == 0:
    print('\nUsing sample GPCR sequences...')
    gpcr_df = pd.DataFrame(SAMPLE_GPCRS)
    gpcr_df.to_csv(gpcr_csv, index=False)
    print(f'Saved {len(gpcr_df)} sample GPCR sequences')

---
## Step 2: Sequence Motif Search for Conserved GPCR Motifs

In [ ]:
# View GPCR motif library
gpcr_motifs_file = os.path.join(MOTIF_LIBRARIES_DIR, 'gpcr_motifs.csv')
gpcr_motifs = pd.read_csv(gpcr_motifs_file)
print('GPCR Motif Library:')
gpcr_motifs

In [ ]:
# Run sequence motif search
original_dir = os.getcwd()
os.chdir('../sequence_motif')

existing_outputs = set(glob.glob('outputs/*_all_results.csv'))

run_motif_search(
    motifs_file=gpcr_motifs_file,
    motif_column='consensus',
    motif_name_column='motif_name',
    sequences_file=gpcr_csv,
    sequence_column='sequence',
    output_file='ignored',
    name_column='Entry'
)

new_outputs = set(glob.glob('outputs/*_all_results.csv')) - existing_outputs
if new_outputs:
    seq_results_file = os.path.abspath(list(new_outputs)[0])
else:
    all_outputs = sorted(glob.glob('outputs/*_all_results.csv'))
    seq_results_file = os.path.abspath(all_outputs[-1]) if all_outputs else None

os.chdir(original_dir)
print(f'\nResults saved to: {seq_results_file}')

In [ ]:
# Analyze sequence motif results
motif_counts = {}
if seq_results_file and os.path.exists(seq_results_file):
    seq_results = pd.read_csv(seq_results_file)
    print(f'Analyzed {len(seq_results)} sequences')
    for idx, row in seq_results.iterrows():
        if pd.notna(row.get('motifs', None)):
            try:
                motifs_data = json.loads(row['motifs'])
                for motif_pattern, motif_info in motifs_data.items():
                    name = motif_info.get('motif_name', motif_pattern)
                    matches = motif_info.get('matches', [])
                    motif_counts[name] = motif_counts.get(name, 0) + len(matches)
            except:
                pass
    print('\nMotif Occurrence Summary:')
    for name, count in sorted(motif_counts.items(), key=lambda x: -x[1]):
        print(f'  {name}: {count} occurrences')

---
## Step 3: Structure Motif Search for Binding Pockets

Search GPCR crystal structures for binding pocket motifs:
- **2RH1** - Beta-2 adrenergic receptor 
- **3SN6** - Beta-2 adrenergic receptor with G-protein
- **4DKL** - Opioid receptor with antagonist

In [ ]:
# Load GPCR binding pocket motif definition
gpcr_pocket_file = os.path.join(STRUCTURE_MOTIFS_DIR, 'gpcr_binding_pocket.json')
with open(gpcr_pocket_file, 'r') as f:
    pocket_motif = json.load(f)

print(f"Motif: {pocket_motif['motif_name']}")
print(f"\nComponents:")
for comp in pocket_motif['components']:
    print(f"  - {comp['id']}: {comp['residue_type']}")

In [ ]:
# Search GPCR structures specifically
gpcr_structures = ['2RH1.pdb', '3SN6.pdb', '4DKL.pdb']
gpcr_results = {}

print('Searching GPCR structures for binding pocket motifs...')
for pdb_name in gpcr_structures:
    pdb_path = os.path.join(PROTEIN_FILES_DIR, pdb_name)
    if os.path.exists(pdb_path):
        try:
            found = search_single_file(pdb_path, pocket_motif)
            gpcr_results[pdb_name] = {'path': pdb_path, 'matches': found, 'count': len(found)}
            print(f"  {pdb_name}: {len(found)} binding pocket(s) found")
        except Exception as e:
            print(f"  {pdb_name}: Error - {e}")
    else:
        print(f"  {pdb_name}: Not found - download from PDB")

---
## Step 4: py3Dmol Visualization of Binding Pockets

Visualize the identified binding pocket residues in 3D using py3Dmol.

In [ ]:
def visualize_binding_pocket(pdb_path, matches, title="GPCR Binding Pocket"):
    """
    Visualize a GPCR structure with binding pocket residues highlighted.
    
    Args:
        pdb_path: Path to PDB file
        matches: List of motif matches from search_single_file
                 Format: [{"residues": [{"res_name": ..., "chain_id": ..., "res_id": ...}, ...]}, ...]
        title: Title for the visualization
    """
    # Read PDB file
    with open(pdb_path, 'r') as f:
        pdb_data = f.read()
    
    # Create viewer
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, 'pdb')
    
    # Style the receptor as cartoon (transparent)
    view.setStyle({'cartoon': {'color': 'lightgray', 'opacity': 0.7}})
    
    # Highlight binding pocket residues from matches
    if matches and len(matches) > 0:
        # Take first match for visualization
        match = matches[0]
        
        # Match format is {"residues": [...]}
        residues = match.get('residues', [])
        
        # Add spheres for each residue in the match
        for residue_info in residues:
            chain = residue_info.get('chain_id', 'A')
            resid = residue_info.get('res_id', 0)
            resname = residue_info.get('res_name', 'UNK')
            
            # Determine color based on residue type
            if resname == 'TRP':
                color = 'magenta'
            elif resname == 'ASP':
                color = 'red'
            elif resname == 'PHE':
                color = 'orange'
            elif resname == 'TYR':
                color = 'yellow'
            else:
                color = 'cyan'
            
            # Highlight residue as sticks and spheres
            view.addStyle(
                {'chain': chain, 'resi': resid},
                {'stick': {'radius': 0.3, 'color': color}, 
                 'sphere': {'radius': 0.8, 'color': color, 'opacity': 0.6}}
            )
    
    # Add ligand if present (common heteroatom style)
    view.addStyle({'hetflag': True}, {'stick': {'radius': 0.25, 'color': 'green'}})
    
    # Center and zoom
    view.zoomTo()
    
    print(f"\n{title}")
    print(f"Binding pocket residues highlighted:")
    print("  - Magenta: TRP (toggle switch)")
    print("  - Red: ASP (anchor)")
    print("  - Orange: PHE (aromatic)")
    print("  - Yellow: TYR (TM7)")
    print("  - Green: Ligand")
    
    return view

In [ ]:
# Visualize 2RH1 - Beta-2 adrenergic receptor
if '2RH1.pdb' in gpcr_results and gpcr_results['2RH1.pdb']['count'] > 0:
    view = visualize_binding_pocket(
        gpcr_results['2RH1.pdb']['path'],
        gpcr_results['2RH1.pdb']['matches'],
        "2RH1: Beta-2 Adrenergic Receptor with Carazolol"
    )
    view.show()
else:
    # Show structure even without matches
    pdb_path = os.path.join(PROTEIN_FILES_DIR, '2RH1.pdb')
    if os.path.exists(pdb_path):
        with open(pdb_path, 'r') as f:
            pdb_data = f.read()
        view = py3Dmol.view(width=800, height=600)
        view.addModel(pdb_data, 'pdb')
        view.setStyle({'cartoon': {'color': 'spectrum'}})
        view.addStyle({'hetflag': True}, {'stick': {'radius': 0.3, 'color': 'green'}})
        view.zoomTo()
        print("2RH1: Beta-2 Adrenergic Receptor (showing full structure)")
        print("  - Rainbow: N to C terminus")
        print("  - Green sticks: Ligand")
        view.show()

In [ ]:
# Visualize 3SN6 - Beta-2 adrenergic with G-protein complex
pdb_path = os.path.join(PROTEIN_FILES_DIR, '3SN6.pdb')
if os.path.exists(pdb_path):
    with open(pdb_path, 'r') as f:
        pdb_data = f.read()
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, 'pdb')
    # Color receptor and G-protein differently
    view.setStyle({'chain': 'R'}, {'cartoon': {'color': 'blue', 'opacity': 0.8}})  # Receptor
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'green', 'opacity': 0.7}}) # Gs alpha
    view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'orange', 'opacity': 0.7}}) # Gs beta
    view.setStyle({'chain': 'G'}, {'cartoon': {'color': 'yellow', 'opacity': 0.7}}) # Gs gamma
    view.addStyle({'hetflag': True}, {'stick': {'radius': 0.3, 'color': 'magenta'}})
    view.zoomTo()
    print("3SN6: Beta-2 Adrenergic Receptor with Gs Protein Complex")
    print("  - Blue: Receptor")
    print("  - Green/Orange/Yellow: G-protein subunits")
    print("  - Magenta: Ligand")
    view.show()

In [ ]:
# Visualize 4DKL - Opioid receptor with antagonist
if '4DKL.pdb' in gpcr_results and gpcr_results['4DKL.pdb']['count'] > 0:
    view = visualize_binding_pocket(
        gpcr_results['4DKL.pdb']['path'],
        gpcr_results['4DKL.pdb']['matches'],
        "4DKL: Dopamine D3 Receptor with Eticlopride"
    )
    view.show()
else:
    pdb_path = os.path.join(PROTEIN_FILES_DIR, '4DKL.pdb')
    if os.path.exists(pdb_path):
        with open(pdb_path, 'r') as f:
            pdb_data = f.read()
        view = py3Dmol.view(width=800, height=600)
        view.addModel(pdb_data, 'pdb')
        view.setStyle({'cartoon': {'color': 'purple'}})
        view.addStyle({'hetflag': True}, {'stick': {'radius': 0.3, 'color': 'green'}})
        view.zoomTo()
        print("4DKL: Dopamine D3 Receptor (showing full structure)")
        print("  - Purple: Receptor")
        print("  - Green sticks: Ligand (eticlopride)")
        view.show()